## Training with a ```Session``` object

In [1]:
import sys
from architecture.session_bigdata import Session, fno_ver1, fno_ver2, fno_ver3, fno_ver4
from architecture.attending_fno_1d import AFNO1d

######### FIX MATRIX PLOTTING TAMALE #####
N_train = 1000
#data_dir = "/home/emastr/deep-micro-slip-model/data/micro_geometries_boundcurv_repar_256_torch/data_big_clean.torch"
data_dir = "/home/emastr/deep-micro-slip-model/data/micro_geometries_boundcurv_repar_256_torch_high_variance/"
#data_dir = "/mnt/data0/emastr/geometries_torch/variance_norepar_512/"
#save_dir = "/mnt/data0/emastr/article_training_nodecay/"
save_dir = "/mnt/data0/emastr/training/article_training_norepar/"
dash_dir = "/home/emastr/deep-micro-slip-model/data/dashboard/attendingfno/"

In [2]:
device="cuda:0"
for seed in [0, 1, 2, 3, 4, 5]:
    
    save_name = f"afno1d_ver3_seed{seed}"
    device = "cuda:0"
    net = AFNO1d(device=device, dtype="float", attention_version=3)
    #print(net.layer_widths)
    session = Session(net, save_name=save_name, save_dir=save_dir, dash_dir=dash_dir, device=device, path_data=data_dir, lr_schedule=True)
    session.train_nsteps(N_train)
    #print(f"Seed {seed}, model ver4")
    #net = bitdonet(device=device) 
    #session = Session(net, save_name=save_name, save_dir=save_dir, dash_dir=dash_dir + "v4", device=device, path_data=data_dir)
    #session.train_nsteps(N_train)
    
    save_name = f"afno1d_seed{seed}"
    device = "cuda:0"
    net = AFNO1d(device=device, dtype="float")
    #print(net.layer_widths)
    session = Session(net, save_name=save_name, save_dir=save_dir, dash_dir=dash_dir, device=device, path_data=data_dir, lr_schedule=False)
    session.train_nsteps(N_train)
    #print(f"Seed {seed}, model ver4")
    #net = bitdonet(device=device) 
    #session = Session(net, save_name=save_name, save_dir=save_dir, dash_dir=dash_dir + "v4", device=device, path_data=data_dir)
    #session.train_nsteps(N_train)

#### Some speed Tests

In [ ]:
import torch
from matplotlib import pyplot as plt
from architecture.attending_fno_1d import AFNO1d, Attention, SpectralConv1d

def from_compl(x):
    out = torch.zeros(x.size(0), 2*x.size(1), x.size(-1), device=x.device, dtype=x.dtype)
    for i in range(x.size(1)):
        out[:, 2*i] = x[:, i].real
        out[:, 2*i+1] = x[:, i].imag
    return out

def to_compl(x):
    return x[:, ::2] + 1j*x[:, 1::2]

def compl_mult(a, x):
    return from_compl(a*to_compl(x))

save_name = f"afno1d_seed{0}"
device = "cuda:0"
net = AFNO1d(device=device, dtype="float")
session = Session(net, save_name=save_name, save_dir=save_dir, dash_dir=dash_dir, device=device, path_data=data_dir, lr_schedule=False)
inp_f = lambda x: torch.einsum("ij, bjx -> bix", net.inp, to_compl(x))

a = torch.exp(1j * torch.Tensor([torch.pi])/4).to(device, dtype=torch.cfloat)
x = session.data.X[:100]
x_compl = to_compl(x)


# Check invariance.
print(torch.linalg.norm(compl_mult(a, net(x)) - net(compl_mult(a, x)))/torch.linalg.norm(net(x)))

def truncfft2(x, n):
    xfft = SpectralConv1d.truncfft(x, n)
    return SpectralConv1d.itruncfft(xfft, x.size(-1), n)

plt.figure()
x_compl_fft = torch.fft.fft(x_compl, dim=-1)
plt.plot(torch.abs(x_compl_fft[0,0]).cpu().numpy()) 
plt.plot(torch.abs(torch.roll(torch.roll(x_compl_fft[0,0], 20)[:41], -20)).cpu().numpy())
plt.yscale("log")

plt.figure()
modes = 41
x_compl_fft_ifft = truncfft2(x_compl, modes)
#x_compl_fft_ifft = truncfft(x_compl,modes)
plt.plot(x_compl_fft_ifft[0,0].real.cpu().numpy(), x_compl_fft_ifft[0,0].imag.cpu().numpy())
plt.plot(x_compl[0,0].real.cpu().numpy(), x_compl[0,0].imag.cpu().numpy())

